### Text Preprocessing

#### import libraries

In [25]:
import pandas as pd
import numpy as np
import re

from nltk.corpus import stopwords
import nltk

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [26]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/aximsoft/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Load the datasets

In [3]:
data=pd.read_csv('../dataset/IMDB Dataset.csv')

### Missing Vlaues

In [4]:
print(data.isnull().sum())

review       0
sentiment    0
dtype: int64


##### No missing values in the datasets

### Handle Duplicates

In [5]:
print("Duplicate rows:", data.duplicated().sum())

Duplicate rows: 418


In [6]:
duplicates=data[data['review'].duplicated(keep=False)]
duplicates.head(20)

,review,sentiment
42,"Of all the films I have seen, this one, The Ra...",negative
84,"We brought this film as a joke for a friend, a...",negative
140,"Before I begin, let me get something off my ch...",negative
219,Ed Wood rides again. The fact that this movie ...,negative
245,I have seen this film at least 100 times and I...,positive
480,From director Barbet Schroder (Reversal of For...,negative
513,"The story and the show were good, but it was r...",negative
636,I rented this thinking it would be pretty good...,negative
638,This movie has everything typical horror movie...,positive
701,I Enjoyed Watching This Well Acted Movie Very ...,positive


In [7]:
duplicates["sentiment"].value_counts()

sentiment
negative    598
positive    226
Name: count, dtype: int64

In [9]:
print("Before removing duplicates:")
print(data["sentiment"].value_counts())

print(
    data["sentiment"].value_counts(normalize=True) * 100
)

Before removing duplicates:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
sentiment
positive    50.0
negative    50.0
Name: proportion, dtype: float64


In [11]:
df_no_duplicates = data.drop_duplicates()

In [12]:
print("After removing duplicates:")
print(df_no_duplicates["sentiment"].value_counts())

print(
    df_no_duplicates["sentiment"].value_counts(normalize=True) * 100
)

After removing duplicates:
sentiment
positive    24884
negative    24698
Name: count, dtype: int64
sentiment
positive    50.187568
negative    49.812432
Name: proportion, dtype: float64


In [16]:
data = data.drop_duplicates(subset=["review"]).copy()

In [17]:
print("Dataset shape:", data.shape)
print("Duplicates:", data["review"].duplicated().sum())

Dataset shape: (49582, 2)
Duplicates: 0


### Convert sentiment into numbers

In [19]:
data["label"] = data["sentiment"].map({"negative": 0,"positive":1})

In [20]:
data[["sentiment", "label"]].head()

,sentiment,label
0,positive,1
1,positive,1
2,positive,1
3,negative,0
4,positive,1


### Create the text preprocessing function

In [27]:
stop_words = set(stopwords.words("english"))

In [28]:
def preprocess_text(text):
    # convert to lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    # Remove punctuation and unnecessary characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    # Handle stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]
    # Join words back together
    text = " ".join(words)
    return text

### Test the preprocessing function

In [29]:
sample_review = data["review"].iloc[0]
print("Original:")
print(sample_review)
print("\nPreprocessed:")
print(preprocess_text(sample_review))

Original:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due 

### Apply preprocessing to all reviews

In [30]:
data["cleaned_review"] = data["review"].apply(preprocess_text)

In [31]:
data[["review","cleaned_review"]].head()

,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one reviewers mentioned watching oz episode ho...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake thinks zombie...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


In [33]:
data.head()

,review,sentiment,label,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,1,one reviewers mentioned watching oz episode ho...
1,A wonderful little production. <br /><br />The...,positive,1,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,1,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,0,basically family little boy jake thinks zombie...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1,petter mattei love time money visually stunnin...


#### Seperate X and y

In [32]:
X = data["cleaned_review"]
y = data["label"]

#### Create Train, Validation and Test datasets

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)

In [35]:
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train,test_size=0.1765,random_state=42,stratify=y_train)

In [36]:
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Training: 34705
Validation: 7439
Testing: 7438


### Tokenization

In [37]:
vocab_size = 20000
tokenizer = Tokenizer(num_words=vocab_size,oov_token="<OOV>")

In [38]:
tokenizer.fit_on_texts(X_train)

### Convert text into sequences

In [39]:
#Training
X_train_sequences = tokenizer.texts_to_sequences(X_train)
#Validation
X_val_sequences = tokenizer.texts_to_sequences(X_val)
#Testing
X_test_sequences = tokenizer.texts_to_sequences(X_test)

In [40]:
print(X_train.iloc[0])
print(X_train_sequences[0])

much say one except probably worst early spate zombie movies may get watch another one revolt zombies month star john carradine intention building army service third reich seen much james baskett uncle remus song south plays leader also serves carradine manservant black comic mantan moreland reprises fraidy cat chauffeur role king zombies exotically named madame sul te wan carradine housekeeper unfortunately carradine supreme achievement zombification wife brings sorts trouble relatives turn remote abode lab inquire sudden death means fake funeral service actually proves disobedient indignant eventually persuading fellow zombies rise master also involved cowboy star bob steele still best known bit howard hawks big sleep plays u secret agent posing nazi posing sheriff thankfully director sekely would much better luck next genre effort day triffids
[14, 49, 4, 430, 128, 142, 289, 1, 823, 25, 94, 16, 29, 62, 4, 7911, 1116, 3148, 204, 191, 4360, 3438, 1178, 1075, 2165, 741, 18543, 32, 14, 

### Padding

In [41]:
max_length = 200

In [43]:
#Train data
X_train_padded = pad_sequences(X_train_sequences,maxlen=max_length,padding="post",truncating="post")
#Validation
X_val_padded = pad_sequences(X_val_sequences,maxlen=max_length,padding="post",truncating="post")
#Test
X_test_padded = pad_sequences(X_test_sequences,maxlen=max_length,padding="post",truncating="post")

In [44]:
print("Training shape:", X_train_padded.shape)
print("Validation shape:", X_val_padded.shape)
print("Testing shape:", X_test_padded.shape)

Training shape: (34705, 200)
Validation shape: (7439, 200)
Testing shape: (7438, 200)


### Check a padded sequence

In [45]:
print(X_train_padded[0])

[   14    49     4   430   128   142   289     1   823    25    94    16
    29    62     4  7911  1116  3148   204   191  4360  3438  1178  1075
  2165   741 18543    32    14   472     1  1621     1   439  1107   185
  1613    19  2386  4360     1   202   573     1     1 14455     1  1028
 13716   105   550  1116     1   632 12225     1 12454  7288  4360  9447
   349  4360  5736  3340     1   201   812  2419   929  4579   340  2754
     1  3518     1  2025   211   684   991  3683  2165    64  1483     1
     1   714     1  1474  1116  2042   980    19   452  2719   204  1675
  7816    48    39   414   118  1588 11467    87  1500   185   968   894
  1291  6251  2452  6251  1958  2285    60     1     9    14    45  1879
   243   372   633   140     1     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0   

In [48]:
tokenizer.word_index

{'<OOV>': 1,
 'movie': 2,
 'film': 3,
 'one': 4,
 'like': 5,
 'good': 6,
 'time': 7,
 'even': 8,
 'would': 9,
 'story': 10,
 'really': 11,
 'see': 12,
 'well': 13,
 'much': 14,
 'bad': 15,
 'get': 16,
 'great': 17,
 'people': 18,
 'also': 19,
 'first': 20,
 'made': 21,
 'make': 22,
 'could': 23,
 'way': 24,
 'movies': 25,
 'think': 26,
 'characters': 27,
 'character': 28,
 'watch': 29,
 'films': 30,
 'two': 31,
 'seen': 32,
 'many': 33,
 'love': 34,
 'acting': 35,
 'plot': 36,
 'never': 37,
 'life': 38,
 'best': 39,
 'show': 40,
 'know': 41,
 'little': 42,
 'ever': 43,
 'man': 44,
 'better': 45,
 'end': 46,
 'scene': 47,
 'still': 48,
 'say': 49,
 'scenes': 50,
 'something': 51,
 'go': 52,
 'back': 53,
 'real': 54,
 'thing': 55,
 'watching': 56,
 'actors': 57,
 'years': 58,
 'though': 59,
 'director': 60,
 'funny': 61,
 'another': 62,
 'old': 63,
 'actually': 64,
 'work': 65,
 'makes': 66,
 'nothing': 67,
 'look': 68,
 'going': 69,
 'find': 70,
 'lot': 71,
 'new': 72,
 'every': 73,
 'p

### Final labels

In [46]:
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

In [47]:
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(34705,)
(7439,)
(7438,)


In [49]:
import pickle
with open("../models/tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

In [50]:
import numpy as np
np.save("../models/X_train_padded.npy", X_train_padded)
np.save("../models/X_val_padded.npy", X_val_padded)
np.save("../models/X_test_padded.npy", X_test_padded)
np.save("../models/y_train.npy", y_train)
np.save("../models/y_val.npy", y_val)
np.save("../models/y_test.npy", y_test)